# CICIoT2023: class imbalance, SMOTE and feature selection (run everything)

Runs the whole project on Kaggle and packs the results into **`results.zip`** (Output tab, right side).

Before running:
1. **Add Data** → search `CICIoT2023` → add the dataset with the CSV files (`part-00000-….csv` …). Attach only this one dataset.
2. **Settings** (right panel) → **Internet: On** (needs a phone-verified Kaggle account).
3. **Save Version → Save & Run All (Commit)**. It keeps running if you close the tab.

Each phase writes `results.zip` as soon as it finishes, so a timeout still leaves usable results.

In [ ]:
# Settings. 3% of ~46M rows is ~1.4M rows; enough for every class and fast enough for one session.
FRAC = 0.03
SEEDS = "0 1 2"          # seeds for the main comparison and the feature sweep
BRANCH = "main"
REPO = "https://github.com/Dhruvsandilya0711/IoT_Mini_Project.git"
WORK = "/kaggle/working"

## 1. Get the project code

In [ ]:
import os, subprocess, glob, shutil
os.chdir(WORK)
if not os.path.isdir("IoT_Mini_Project"):
    ok = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO]).returncode == 0
    if not ok:
        # Fallback without Internet: the repo uploaded as a Kaggle dataset (zip of the GitHub branch).
        found = glob.glob("/kaggle/input/**/src/train.py", recursive=True)
        assert found, "No Internet and no uploaded copy of the repo. Turn Internet on in Settings."
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), "IoT_Mini_Project")
os.chdir(f"{WORK}/IoT_Mini_Project")
print(os.getcwd(), os.listdir("src"))

In [ ]:
!pip install -q "imbalanced-learn>=0.12" "pyarrow>=14" 2>&1 | tail -2
import sklearn, imblearn, pandas
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__, "| pandas", pandas.__version__)

## 2. Find the CICIoT2023 files

In [ ]:
import pandas as pd
from collections import Counter

import pyarrow.parquet as pq

def has_label(path):
    try:
        cols = pd.read_csv(path, nrows=0).columns if path.endswith(".csv") else pq.read_schema(path).names
        return any(c.strip().lower() == "label" for c in cols)
    except Exception:
        return False

INPUT = "/kaggle/input"
files = [p for p in glob.glob(f"{INPUT}/**/*", recursive=True)
         if p.lower().endswith((".csv", ".parquet")) and "IoT_Mini_Project" not in p]
data_files = [p for p in files if has_label(p)]
by_dir = Counter(os.path.dirname(p) for p in data_files)
print("folders with labelled CICIoT2023 files:", by_dir.most_common(5))
assert by_dir, "No CICIoT2023 files found. Add the dataset with Add Data (search CICIoT2023)."

# Use the folder with the most parts, so merged copies elsewhere are not read twice.
raw_src = by_dir.most_common(1)[0][0]
chosen = sorted(p for p in data_files if os.path.dirname(p) == raw_src)
RAW = f"{WORK}/raw"
shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW)
for p in chosen:
    os.symlink(p, os.path.join(RAW, os.path.basename(p)))
size_gb = sum(os.path.getsize(p) for p in chosen) / 1e9
print(f"Using {len(chosen)} files ({size_gb:.1f} GB) from {raw_src}")

## 3. Sample, split and explore the data

In [ ]:
!python -m src.load --raw-dir {RAW} --frac {FRAC} 2>&1 | tail -12
!python -m src.preprocess
!python -m src.eda 2>&1 | tail -15

In [ ]:
def pack(note):
    os.makedirs("results/data_info", exist_ok=True)
    for f in glob.glob("data/processed/*.json"):
        shutil.copy(f, "results/data_info/")
    shutil.make_archive(f"{WORK}/results", "zip", "results")
    print(f"results.zip updated after: {note}")
pack("EDA")

## 4. Reproduction: baseline Decision Tree and Random Forest (8, 34, 2 classes)

In [ ]:
!bash scripts/run_experiments.sh baseline 2>&1 | grep -E "^==|^test:|skipping|Error|error"
pack("baseline")

## 5. Improvement: SMOTE, SMOTE + undersampling and class weights vs baseline

In [ ]:
os.environ["SEEDS"] = SEEDS
!bash scripts/run_experiments.sh improvement 2>&1 | grep -E "^==|^test:|skipping|Error|error|Baseline|SMOTE|Class weights"
pack("improvement")

## 6. Extension: feature selection for lightweight gateways

In [ ]:
!bash scripts/run_experiments.sh extension 2>&1 | grep -E "^==|^test:|Ranked|skipping|Error|error"
pack("extension")

## 7. Summary

In [ ]:
for f in sorted(glob.glob("results/tables/*_summary.csv")) + sorted(glob.glob("results/tables/sweep_*.csv")):
    print("\n==", f)
    t = pd.read_csv(f)
    cols = [c for c in ["strategy", "n_features", "seeds", "test_accuracy", "test_macro_f1", "test_minority_recall",
                        "test_worst_class_recall", "predict_us_per_flow", "model_size_mb", "train_s"] if c in t]
    print(t[cols].to_string(index=False, float_format="%.4f"))
print("\nDownload results.zip from the Output tab and unzip it into the repository's results/ folder.")